In [1]:
"""
PHASE 1: Data Structure Design and Implementation
Application Context: E-commerce Recommendation System
----------------------------------------------------
This system uses three core data structures:
1. Graph (User–Item Relationship Network)
2. Hash Table (User–Item Interaction Lookup)
3. Heap (Top-K Product Recommendations)
"""

import heapq
from collections import defaultdict

# ------------------------------------------------------------
# 1. GRAPH DATA STRUCTURE: User–Item Relationship Graph
# ------------------------------------------------------------
class UserItemGraph:
    """
    A bipartite graph representing users and items.
    Each user connects to multiple items they interacted with (viewed, purchased, rated).
    """

    def __init__(self):
        self.graph = defaultdict(set)

    def add_interaction(self, user, item):
        """Add an edge between a user and an item."""
        self.graph[user].add(item)
        self.graph[item].add(user)

    def get_connected_items(self, user):
        """Return all items a given user is connected to."""
        return [node for node in self.graph[user] if node.startswith("Item")]

    def get_connected_users(self, item):
        """Return all users connected to an item."""
        return [node for node in self.graph[item] if node.startswith("User")]

    def get_all_users(self):
        return [node for node in self.graph if node.startswith("User")]

    def get_all_items(self):
        return [node for node in self.graph if node.startswith("Item")]


# ------------------------------------------------------------
# 2. HASH TABLE DATA STRUCTURE: User–Item Interaction Map
# ------------------------------------------------------------
class InteractionHashTable:
    """
    Custom hash table implementation for storing user-item interactions.
    Used for quick lookup of preferences or ratings.
    """

    def __init__(self, size=100):
        self.size = size
        self.table = [[] for _ in range(size)]

    def _hash(self, key):
        """Hash function using simple modulo."""
        return hash(key) % self.size

    def insert(self, user_item_key, rating):
        """Insert or update a user-item rating."""
        index = self._hash(user_item_key)
        for i, (key, value) in enumerate(self.table[index]):
            if key == user_item_key:
                self.table[index][i] = (key, rating)
                return
        self.table[index].append((user_item_key, rating))

    def get(self, user_item_key):
        """Retrieve the rating for a given user-item pair."""
        index = self._hash(user_item_key)
        for key, value in self.table[index]:
            if key == user_item_key:
                return value
        return None

    def __str__(self):
        """Pretty-print hash table contents."""
        result = []
        for i, bucket in enumerate(self.table):
            if bucket:
                result.append(f"Bucket {i}: {bucket}")
        return "\n".join(result)


# ------------------------------------------------------------
# 3. HEAP DATA STRUCTURE: Top-K Product Recommendations
# ------------------------------------------------------------
class TopKHeap:
    """
    Min-heap implementation to maintain top-K recommended items for a user.
    """

    def __init__(self, k):
        self.k = k
        self.heap = []

    def add(self, item, score):
        """Add item with a score; keep only top-K."""
        if len(self.heap) < self.k:
            heapq.heappush(self.heap, (score, item))
        else:
            heapq.heappushpop(self.heap, (score, item))

    def get_top_k(self):
        """Return sorted top-K items."""
        return sorted(self.heap, reverse=True)

    def __str__(self):
        return str(self.get_top_k())


# ------------------------------------------------------------
# 4. SAMPLE WORKFLOW / DEMONSTRATION
# ------------------------------------------------------------
if __name__ == "__main__":

    print("===== E-COMMERCE RECOMMENDATION SYSTEM DEMO =====")

    # Initialize Graph and Hash Table
    user_item_graph = UserItemGraph()
    interaction_map = InteractionHashTable(size=10)

    # Step 1: Add interactions (user-item relationships)
    interactions = [
        ("User1", "ItemA", 5),
        ("User1", "ItemB", 4),
        ("User2", "ItemA", 4),
        ("User2", "ItemC", 3),
        ("User3", "ItemB", 5),
        ("User3", "ItemC", 4),
        ("User4", "ItemD", 5),
    ]

    for user, item, rating in interactions:
        user_item_graph.add_interaction(user, item)
        interaction_map.insert(f"{user}-{item}", rating)

    # Step 2: Display stored relationships
    print("\nGraph Connections:")
    for user in user_item_graph.get_all_users():
        print(f"{user} -> {user_item_graph.get_connected_items(user)}")

    print("\nHash Table (User–Item Ratings):")
    print(interaction_map)

    # Step 3: Simulate Recommendation Scoring
    print("\nGenerating recommendations for User1...")
    heap = TopKHeap(k=3)

    # Example: Suppose system computes similarity scores for candidate items
    candidate_scores = {
        "ItemA": 0.9,
        "ItemB": 0.8,
        "ItemC": 0.75,
        "ItemD": 0.6,
        "ItemE": 0.95,
    }

    for item, score in candidate_scores.items():
        heap.add(item, score)

    print("\nTop 3 Recommended Items for User1:")
    for score, item in heap.get_top_k():
        print(f"{item} (score: {score})")

    # Step 4: Lookup specific rating
    key = "User1-ItemB"
    print(f"\nRating lookup for {key}: {interaction_map.get(key)}")

    print("\n===== END OF DEMO =====")


===== E-COMMERCE RECOMMENDATION SYSTEM DEMO =====

Graph Connections:
User1 -> ['ItemB', 'ItemA']
User2 -> ['ItemA', 'ItemC']
User3 -> ['ItemB', 'ItemC']
User4 -> ['ItemD']

Hash Table (User–Item Ratings):
Bucket 1: [('User1-ItemB', 4)]
Bucket 3: [('User2-ItemA', 4)]
Bucket 4: [('User1-ItemA', 5), ('User2-ItemC', 3)]
Bucket 6: [('User3-ItemC', 4)]
Bucket 8: [('User4-ItemD', 5)]
Bucket 9: [('User3-ItemB', 5)]

Generating recommendations for User1...

Top 3 Recommended Items for User1:
ItemE (score: 0.95)
ItemA (score: 0.9)
ItemB (score: 0.8)

Rating lookup for User1-ItemB: 4

===== END OF DEMO =====
